In [ ]:
#Complete Code for "Order Delivery Time Prediction using Machine
Learning in Python"
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from sklearn import svm
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Load dataset
dataset = pd.read_csv("talabat_enhanced_orders.csv")
print(dataset.head(5))
print(dataset.shape)

# Data type categorization
obj = (dataset.dtypes == 'object')
object_cols = list(obj[obj].index)
int_ = (dataset.dtypes == 'int64')
num_cols = list(int_[int_].index)
fl_ = (dataset.dtypes == 'float64')
fl_cols = list(fl_[fl_].index)

# Heatmap
numerical_dataset = dataset.select_dtypes(include=['int64', 'float64'])
plt.figure(figsize=(12, 6))
sns.heatmap(numerical_dataset.corr(), cmap='BrBG', fmt='.2f', linewidths=2,
annot=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.tight_layout()
plt.savefig("correlation_heatmap.png")

# Bar plot - unique values
unique_values = []
for col in object_cols:
    unique_values.append(dataset[col].unique().size)
plt.figure(figsize=(10,6))
sns.barplot(x=object_cols, y=unique_values)

# Bar plot - distribution
plt.figure(figsize=(18, 16))
index = 1
for col in object_cols:
    y = dataset[col].value_counts()
    plt.subplot(3, 3, index)
    sns.barplot(x=list(y.index), y=y)
    index += 1
plt.tight_layout()

# Data Cleaning
drop_cols = ['User_ID', 'Item_Name', 'Order_Time', 'Delivery_Time',
             'Restaurant_Lat', 'Restaurant_Lon', 'Customer_Lat', 'Customer_Lon',
             'Driver_Lat', 'Driver_Lon', 'Order_ID']
dataset.drop(drop_cols, axis=1, inplace=True)
dataset['Delivery_Duration_Minutes'] = dataset['Delivery_Duration_Minutes'].fillna(
    dataset['Delivery_Duration_Minutes'].mean())
new_dataset = dataset.dropna()

# OneHotEncoding
s = (new_dataset.dtypes == 'object')
object_cols = list(s[s].index)
OH_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
OH_cols = pd.DataFrame(OH_encoder.fit_transform(new_dataset[object_cols]))
OH_cols.index = new_dataset.index
OH_cols.columns = OH_encoder.get_feature_names_out()
df_final = new_dataset.drop(object_cols, axis=1)
df_final = pd.concat([df_final, OH_cols], axis=1)

# Train-Test Split
X = df_final.drop(['Delivery_Duration_Minutes'], axis=1)
Y = df_final['Delivery_Duration_Minutes']
X_train, X_valid, Y_train, Y_valid = train_test_split(
    X, Y, train_size=0.8, test_size=0.2, random_state=0)

# SVM
model_SVR = svm.SVR()
model_SVR.fit(X_train, Y_train)
Y_pred = model_SVR.predict(X_valid)
print(mean_absolute_percentage_error(Y_valid, Y_pred))

# Random Forest
model_RFR = RandomForestRegressor(n_estimators=10)
model_RFR.fit(X_train, Y_train)
Y_pred = model_RFR.predict(X_valid)
print(mean_absolute_percentage_error(Y_valid, Y_pred))

# Linear Regression
model_LR = LinearRegression()
model_LR.fit(X_train, Y_train)
Y_pred = model_LR.predict(X_valid)
print(mean_absolute_percentage_error(Y_valid, Y_pred))